# Pré Processamento dos Dados

## Bibliotecas

In [17]:
import os
from pathlib import Path
from pysus import query_parquet
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import duckdb

## Caminhos dos Arquivos para Queries

In [18]:
# configuração dos caminhos para as queries usando duckdb

# diretório base do repo
BASE_DIR = (
    Path().resolve().parent 
    if Path().resolve().name == "notebooks" 
    else Path().resolve()
)

# aponta para diretório com dados brutos do SIH
SIH_DIR = BASE_DIR / 'data' / 'pysus_cache' / 'downloads' / 'ducklake' / 'sih'
# as_posix() para caminho glob (string literal, não objeto como pathlib)
sih_glob = (SIH_DIR / '*.parquet').as_posix()

#print(f"Padrão de busca para o DuckDB: {sih_glob}")

## Verificação de Valores Únicos Para Cada Coluna

Muitas colunas não têm informações relevantes. Para selecionar as variáveis adequadas para o estudo, preciso investigar primeiramente a quantidade de valores únicos, o que proporciona uma análise mais direta da utilidade das variáveis.

In [20]:
# estabelecendo conexão com duckdb
con = duckdb.connect()

# gera o perfil de todas as 114 colunas
df_perfil = con.execute(f"SUMMARIZE SELECT * FROM '{sih_glob}'").df()

# ordena as colunas do df para verificar o perfilamento do conjunto de dados
df_analise = df_perfil[['column_name', 'column_type', 'null_percentage', 'approx_unique']]

# tirando o limite de linhas e colunas da exibição do DF do pandas
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    
    df_ordenado = df_analise.sort_values(
        by=['null_percentage', 'approx_unique'], 
        ascending=[False, False]
    )
    
    display(df_ordenado)

,column_name,column_type,null_percentage,approx_unique
5,N_AIH,VARCHAR,0.0,2144336
35,VAL_TOT,VARCHAR,0.0,426686
24,VAL_SH,VARCHAR,0.0,363506
37,US_TOT,VARCHAR,0.0,173416
25,VAL_SP,VARCHAR,0.0,73208
7,CEP,VARCHAR,0.0,50637
9,NASC,VARCHAR,0.0,42297
85,SEQUENCIA,VARCHAR,0.0,32965
65,INSC_PN,VARCHAR,0.0,22339
40,DIAG_PRINC,VARCHAR,0.0,8048


## Exploração das Colunas

In [21]:
# consulta da frequência dos códigos da coluna ESPEC (Especialidade)
query_espec = f"""
    SELECT 
        ESPEC,
        COUNT(*) AS total_internacoes,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_total
    FROM '{sih_glob}'
    GROUP BY ESPEC
    ORDER BY total_internacoes DESC
"""

df_espec = con.execute(query_espec).df()
display(df_espec.head(10))

,ESPEC,total_internacoes,pct_total
0,01,797735,40.10
1,03,754763,37.94
2,02,181785,9.14
3,07,123422,6.20
4,05,66278,3.33
5,87,23444,1.18
6,09,21866,1.10
7,04,16732,0.84
8,10,1346,0.07
9,12,611,0.03


In [27]:
# consulta da frequência dos códigos da coluna UF_ZI
query_uf_zi = f"""
    SELECT 
        UF_ZI,
        COUNT(*) AS total_internacoes,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_total
    FROM '{sih_glob}'
    GROUP BY UF_ZI
    ORDER BY total_internacoes DESC
"""

df_uf_zi = con.execute(query_uf_zi).df()
display(df_uf_zi.head(10))

,UF_ZI,total_internacoes,pct_total
0,430000,932552,46.88
1,431490,429011,21.57
2,430510,71880,3.61
3,430460,58031,2.92
4,431440,55253,2.78
5,431340,30564,1.54
6,431680,30357,1.53
7,431870,28664,1.44
8,431720,22484,1.13
9,431140,20981,1.05


In [28]:
# consulta da frequência dos códigos da coluna RACA_COR
query_raca_cor = f"""
    SELECT 
        RACA_COR,
        COUNT(*) AS total_internacoes,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_total
    FROM '{sih_glob}'
    GROUP BY RACA_COR
    ORDER BY total_internacoes DESC
"""

df_raca_cor = con.execute(query_raca_cor).df()
display(df_raca_cor.head(10))

,RACA_COR,total_internacoes,pct_total
0,01,1700035,85.46
1,03,154591,7.77
2,02,114090,5.74
3,04,13902,0.70
4,05,6529,0.33
5,99,24,0.00


In [29]:
# consulta da frequência dos códigos da coluna INSTRU
query_instru = f"""
    SELECT 
        INSTRU,
        COUNT(*) AS total_internacoes,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_total
    FROM '{sih_glob}'
    GROUP BY INSTRU
    ORDER BY total_internacoes DESC
"""

df_instru = con.execute(query_instru).df()
display(df_instru.head(10))

,INSTRU,total_internacoes,pct_total
0,0,1963454,98.71
1,3,13250,0.67
2,2,8799,0.44
3,4,2192,0.11
4,1,1476,0.07


In [30]:
# consulta da frequência dos códigos da coluna MORTE
query_morte = f"""
    SELECT 
        MORTE,
        COUNT(*) AS total_internacoes,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_total
    FROM '{sih_glob}'
    GROUP BY MORTE
    ORDER BY total_internacoes DESC
"""

df_morte = con.execute(query_morte).df()
display(df_morte.head(10))

,MORTE,total_internacoes,pct_total
0,0,1886099,94.82
1,1,103072,5.18


In [32]:
# COMPLEX
# consulta da frequência dos códigos da coluna COMPLEX
query_complex = f"""
    SELECT 
        COMPLEX,
        COUNT(*) AS total_internacoes,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_total
    FROM '{sih_glob}'
    GROUP BY COMPLEX
    ORDER BY total_internacoes DESC
"""

df_complex = con.execute(query_complex).df()
display(df_complex.head(10))

,COMPLEX,total_internacoes,pct_total
0,02,1750206,87.99
1,03,238965,12.01


## Etc

In [7]:
con = duckdb.connect()

# Retorna visão geral de todas as colunas
df_resumo = con.execute(f"""
    SUMMARIZE SELECT * FROM '{sih_glob}'
""").df()

# Filtra o resumo para visualizar colunas-chave de interesse
colunas_chave = ['ESPEC', 'SEXO', 'MARCA_UTI', 'DIAG_PRINC', 'VAL_TOT', 'MORTE']
df_resumo[df_resumo['column_name'].isin(colunas_chave)][
    ['column_name', 'column_type', 'null_percentage', 'approx_unique', 'min', 'max']
]

,column_name,column_type,null_percentage,approx_unique,min,max
3,ESPEC,VARCHAR,0.0,12,01,87
10,SEXO,VARCHAR,0.0,3,1,4
15,MARCA_UTI,VARCHAR,0.0,11,00,99
35,VAL_TOT,VARCHAR,0.0,426686,0.00,9999.95
40,DIAG_PRINC,VARCHAR,0.0,8048,A00,Z998
52,MORTE,VARCHAR,0.0,2,0,1


In [8]:
# Consulta a frequência dos códigos da coluna ESPEC (Especialidade)
query_espec = f"""
    SELECT 
        ESPEC,
        COUNT(*) AS total_internacoes,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_total
    FROM '{sih_glob}'
    GROUP BY ESPEC
    ORDER BY total_internacoes DESC
"""

df_espec = con.execute(query_espec).df()
display(df_espec.head(10))

,ESPEC,total_internacoes,pct_total
0,01,797735,40.10
1,03,754763,37.94
2,02,181785,9.14
3,07,123422,6.20
4,05,66278,3.33
5,87,23444,1.18
6,09,21866,1.10
7,04,16732,0.84
8,10,1346,0.07
9,12,611,0.03


In [14]:
display(df_analise.sort_values(by=['null_percentage', 'approx_unique'], ascending=[False, False]))

,column_name,column_type,null_percentage,approx_unique
5,N_AIH,VARCHAR,0.0,2144336
35,VAL_TOT,VARCHAR,0.0,426686
24,VAL_SH,VARCHAR,0.0,363506
37,US_TOT,VARCHAR,0.0,173416
25,VAL_SP,VARCHAR,0.0,73208
...,...,...,...,...
78,CID_MORTE,VARCHAR,0.0,1
102,DIAGSEC8,VARCHAR,0.0,1
103,DIAGSEC9,VARCHAR,0.0,1
111,TPDISEC8,VARCHAR,0.0,1


## Seleção de Colunas de Interesse

In [8]:
# exploração das colunas presentes no conjunto de dados
query = f"""
    SELECT * FROM '{datasus_path}' LIMIT 10
"""

con.execute(query).df().columns.tolist()

['UF_ZI',
 'ANO_CMPT',
 'MES_CMPT',
 'ESPEC',
 'CGC_HOSP',
 'N_AIH',
 'IDENT',
 'CEP',
 'MUNIC_RES',
 'NASC',
 'SEXO',
 'UTI_MES_IN',
 'UTI_MES_AN',
 'UTI_MES_AL',
 'UTI_MES_TO',
 'MARCA_UTI',
 'UTI_INT_IN',
 'UTI_INT_AN',
 'UTI_INT_AL',
 'UTI_INT_TO',
 'DIAR_ACOM',
 'QT_DIARIAS',
 'PROC_SOLIC',
 'PROC_REA',
 'VAL_SH',
 'VAL_SP',
 'VAL_SADT',
 'VAL_RN',
 'VAL_ACOMP',
 'VAL_ORTP',
 'VAL_SANGUE',
 'VAL_SADTSR',
 'VAL_TRANSP',
 'VAL_OBSANG',
 'VAL_PED1AC',
 'VAL_TOT',
 'VAL_UTI',
 'US_TOT',
 'DT_INTER',
 'DT_SAIDA',
 'DIAG_PRINC',
 'DIAG_SECUN',
 'COBRANCA',
 'NATUREZA',
 'NAT_JUR',
 'GESTAO',
 'RUBRICA',
 'IND_VDRL',
 'MUNIC_MOV',
 'COD_IDADE',
 'IDADE',
 'DIAS_PERM',
 'MORTE',
 'NACIONAL',
 'NUM_PROC',
 'CAR_INT',
 'TOT_PT_SP',
 'CPF_AUT',
 'HOMONIMO',
 'NUM_FILHOS',
 'INSTRU',
 'CID_NOTIF',
 'CONTRACEP1',
 'CONTRACEP2',
 'GESTRISCO',
 'INSC_PN',
 'SEQ_AIH5',
 'CBOR',
 'CNAER',
 'VINCPREV',
 'GESTOR_COD',
 'GESTOR_TP',
 'GESTOR_CPF',
 'GESTOR_DT',
 'CNES',
 'CNPJ_MANT',
 'INFEHOSP',
 'C

In [9]:
# colunas pré selecionadas
colunas = [
    'ANO_CMPT',
    'MES_CMPT',
    'ESPEC',
    'CGC_HOSP',
    'N_AIH',
    'IDENT',
    'MUNIC_RES',
    'NASC',
    'SEXO',
    'MARCA_UTI',
    'VAL_TOT',
    'DT_INTER',
    'DT_SAIDA',
    'DIAG_PRINC',
    'COD_IDADE',
    'IDADE',
    'DIAS_PERM',
    'MORTE',
    'INSTRU',
]
colunas_str = (', '.join(x for x in colunas))
print(colunas_str)

ANO_CMPT, MES_CMPT, ESPEC, CGC_HOSP, N_AIH, IDENT, MUNIC_RES, NASC, SEXO, MARCA_UTI, VAL_TOT, DT_INTER, DT_SAIDA, DIAG_PRINC, COD_IDADE, IDADE, DIAS_PERM, MORTE, INSTRU


## Sumarização do Conjunto de Dados

In [11]:
# Substitua com as colunas que você pré-selecionou
#colunas_alvo = "ANO_CMPT, MES_CMPT, SEXO, IDADE, DIAG_PRINC, ESPEC, MARCA_UTI, DIAS_PERM, MORTE, VAL_TOT"

# O comando SUMMARIZE gera o "profile" dos dados instantaneamente
query_resumo = f"""
    SUMMARIZE SELECT {colunas_str}
    FROM '{datasus_path}'
"""

df_resumo = con.execute(query_resumo).df()

# Selecionamos apenas as colunas mais úteis do resumo para não poluir a tela
display(df_resumo[['column_name', 'column_type', 'null_percentage', 'approx_unique', 'min', 'max']])

,column_name,column_type,null_percentage,approx_unique,min,max
0,ANO_CMPT,VARCHAR,0.0,3,2024,2026
1,MES_CMPT,VARCHAR,0.0,12,01,12
2,ESPEC,VARCHAR,0.0,12,01,87
3,CGC_HOSP,VARCHAR,0.0,266,,98714876000102
4,N_AIH,VARCHAR,0.0,2075387,4320104104107,9926300001948
5,IDENT,VARCHAR,0.0,2,1,5
6,MUNIC_RES,VARCHAR,0.0,1214,110002,530010
7,NASC,VARCHAR,0.0,42297,18991230,20260629
8,SEXO,VARCHAR,0.0,3,1,4
9,MARCA_UTI,VARCHAR,0.0,11,00,99


In [5]:
df_resumo.columns

Index(['column_name', 'column_type', 'min', 'max', 'approx_unique', 'avg',
       'std', 'q25', 'q50', 'q75', 'count', 'null_percentage'],
      dtype='object')

In [ ]:
# testando JOIN com diferentes "tabelas" (arquivos .parquet)
con = duckdb.connect()

# 2. Usa f-string (f""") e aspas simples ao redor das variáveis do Python
query = f"""
    SELECT 
        p.ANO_CMPT,
        p.MES_CMPT,
        LPAD(CAST(p.MUNIC_RES AS VARCHAR), 6, '0') AS codigo_municipio_6d,
        m.nome_municipio,
        m.uf,
        COUNT(*) AS total_internacoes
    FROM '{datasus_path}' AS p
    LEFT JOIN '{ibge_municipios_path}' AS m
        ON LPAD(CAST(p.MUNIC_RES AS VARCHAR), 6, '0') = LPAD(CAST(m.code_ibge_6 AS VARCHAR), 6, '0')
    GROUP BY ALL
    LIMIT 10
"""

df_resultado = con.execute(query).df()
display(df_resultado)

,ANO_CMPT,MES_CMPT,codigo_municipio_6d,nome_municipio,uf,total_internacoes
0,2024,09,432345,Vila Nova do Sul,RS,21
1,2024,09,431310,Nova Palma,RS,59
2,2024,09,431150,Lavras do Sul,RS,33
3,2024,09,430750,Espumoso,RS,64
4,2024,09,430610,Cruz Alta,RS,416
5,2024,09,431535,Quinze de Novembro,RS,34
6,2024,09,430440,Canela,RS,446
7,2024,09,430190,Barra do Ribeiro,RS,74
8,2024,09,430080,Antônio Prado,RS,66
9,2024,09,430620,Cruzeiro do Sul,RS,54


In [12]:
# query exploratória
query = """
    SELECT
        UF_ZI AS municipio_hospital,
        ANO_CMPT AS ano,
        MES_CMPT AS mes,
        DIAS_PERM AS dias_internacao,
        DIAG_PRINC AS diagnostico_cid,
        VAL_TOT AS valor_total
    FROM DATA
    WHERE ANO_CMPT == 2026
    LIMIT 10
"""

query_parquet(caminho_dados, query=query)


┌────────────────────┬─────────┬─────────┬─────────────────┬─────────────────┬─────────────┐
│ municipio_hospital │   ano   │   mes   │ dias_internacao │ diagnostico_cid │ valor_total │
│      varchar       │ varchar │ varchar │     varchar     │     varchar     │   varchar   │
├────────────────────┼─────────┼─────────┼─────────────────┼─────────────────┼─────────────┤
│ 430000             │ 2026    │ 01      │ 4               │ A499            │ 897.91      │
│ 430000             │ 2026    │ 01      │ 8               │ I802            │ 5155.84     │
│ 430000             │ 2026    │ 01      │ 1               │ K564            │ 4275.37     │
│ 430000             │ 2026    │ 01      │ 0               │ T813            │ 543.08      │
│ 430000             │ 2026    │ 01      │ 1               │ D397            │ 413.32      │
│ 430000             │ 2026    │ 01      │ 0               │ G589            │ 785.04      │
│ 430000             │ 2026    │ 01      │ 2               │ N818     

In [ ]:
# verificando colunas
query = """
    SELECT * FROM data LIMIT 10
"""

query_parquet(caminho_dados, query)

┌─────────┬──────────┬──────────┬─────────┬────────────────┬───────────────┬─────────┬──────────┬───────────┬──────────┬─────────┬────────────┬────────────┬────────────┬────────────┬───────────┬────────────┬────────────┬────────────┬────────────┬───────────┬────────────┬────────────┬────────────┬─────────┬─────────┬──────────┬─────────┬───────────┬──────────┬────────────┬────────────┬────────────┬────────────┬────────────┬─────────┬─────────┬─────────┬──────────┬──────────┬────────────┬────────────┬──────────┬──────────┬─────────┬─────────┬─────────┬──────────┬───────────┬───────────┬─────────┬───────────┬─────────┬──────────┬──────────┬─────────┬───────────┬─────────┬──────────┬────────────┬─────────┬───────────┬────────────┬────────────┬───────────┬──────────────┬──────────┬─────────┬─────────┬──────────┬────────────┬───────────┬─────────────────┬───────────┬─────────┬───────────┬──────────┬──────────┬───────────┬─────────┬─────────┬─────────┬─────────┬──────────┬─────────┬──────────

## Exploração das Colunas

In [13]:
# valores únicos de ESPEC
query = f"""
    SELECT ESPEC, COUNT(*) AS Contagem
    FROM '{datasus_path}'
    GROUP BY ESPEC
    ORDER BY ESPEC ASC
"""

df_espec = con.execute(query).df()
df_espec

,ESPEC,Contagem
0,01,768056
1,02,175987
2,03,730014
3,04,16383
4,05,63940
5,06,579
6,07,118842
7,09,20317
8,10,1299
9,12,588


In [16]:
# exploração da MARCA_UTI
query = f"""
    SELECT MARCA_UTI, COUNT(*) AS Contagem
    FROM '{datasus_path}'
    GROUP BY MARCA_UTI
    ORDER BY MARCA_UTI ASC
"""

df_marca_uti = con.execute(query).df()
df_marca_uti

,MARCA_UTI,Contagem
0,00,1814725
1,01,1189
2,75,100361
3,76,35881
4,78,8519
5,79,5389
6,81,15218
7,82,3860
8,83,371
9,85,2262


In [17]:
# exploração da coluna INSTRU
query = f"""
    SELECT INSTRU, COUNT(*)
    FROM '{datasus_path}'
    GROUP BY INSTRU
    ORDER BY INSTRU ASC
"""

df_instru = con.execute(query).df()
df_instru

,INSTRU,count_star()
0,0,1963454
1,1,1476
2,2,8799
3,3,13250
4,4,2192
